# 04 — Teammate Comparison: Pandas/CSV Ingest Pipeline (Phase 1, steps 1-4)


Covers their steps 1-4:
1. Zillow ZHVI ingest/clean
2. Independent verification of data provenance (positive-rate figures vs `docs/phase0_framing.md`)
3. FRED `MORTGAGE30US` ingest/clean
4. Join — monthly Zillow + FRED table

Not included: their steps 5 (per-row coverage audit) and 6 (save output) — out of scope for this comparison.

Their notebook assumes the raw CSVs are already downloaded to disk under specific filenames. Section 0 below adds a download step (not part of their original notebook) since this environment doesn't have those files yet — everything from Section 1 onward mirrors their code as closely as possible.

In [1]:
import pandas as pd

## 0. Download raw data (added for this environment)

The teammate's notebook assumes these files already exist on disk under specific filenames. Downloading them here under those exact paths so the cells below can run unmodified.

In [2]:
import os
import requests

ZILLOW_URL = (
    "https://files.zillowstatic.com/research/public_csvs/zhvi/"
    "County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
)
ZILLOW_RAW_PATH = "../data/raw/zillow/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
os.makedirs(os.path.dirname(ZILLOW_RAW_PATH), exist_ok=True)

if not os.path.exists(ZILLOW_RAW_PATH):
    resp = requests.get(ZILLOW_URL, timeout=60)
    resp.raise_for_status()
    with open(ZILLOW_RAW_PATH, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded {len(resp.content):,} bytes to {ZILLOW_RAW_PATH}")
else:
    print(f"Using cached file at {ZILLOW_RAW_PATH}")

FRED_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
FRED_RAW_PATH = "../data/raw/fred/MORTGAGE30US.csv"
os.makedirs(os.path.dirname(FRED_RAW_PATH), exist_ok=True)

if not os.path.exists(FRED_RAW_PATH):
    resp = requests.get(FRED_URL, timeout=60)
    resp.raise_for_status()
    with open(FRED_RAW_PATH, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded {len(resp.content):,} bytes to {FRED_RAW_PATH}")
else:
    print(f"Using cached file at {FRED_RAW_PATH}")

# The teammate's rename step below matches FRED's columns by name
# ("observation_date", "MORTGAGE30US"). Pin the header here regardless of
# what the fredgraph.csv endpoint actually sends, so their code runs unmodified.
_fred_header = pd.read_csv(FRED_RAW_PATH, nrows=0)
if list(_fred_header.columns) != ["observation_date", "MORTGAGE30US"]:
    _fred_raw = pd.read_csv(FRED_RAW_PATH)
    _fred_raw.columns = ["observation_date", "MORTGAGE30US"]
    _fred_raw.to_csv(FRED_RAW_PATH, index=False)

Using cached file at ../data/raw/zillow/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv
Using cached file at ../data/raw/fred/MORTGAGE30US.csv


## 1. Zillow ZHVI Ingest

Source: County geography, ZHVI Middle Tier (33rd-67th percentile), SFR + Condo/Co-op,
Smoothed & Seasonally Adjusted — downloaded from zillow.com/research/data.
File: County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv (confirmed the actual
file on disk; this is a middle-of-market cut, not Zillow's separate "All Homes"
file).

In [3]:
df = pd.read_csv(ZILLOW_RAW_PATH)

# Sanity check: this must be the county-level file, not metro (see Phase 1 notes
# on the earlier metro/county mix-up)
assert set(df["RegionType"].unique()) == {"county"}, "Not a county-only file!"
print("Raw shape:", df.shape)

Raw shape: (3071, 329)


In [4]:
# Build the county FIPS join key: StateCodeFIPS (2-digit) + MunicipalCodeFIPS (3-digit)
df["county_fips"] = (
    df["StateCodeFIPS"].astype(str).str.zfill(2)
    + df["MunicipalCodeFIPS"].astype(str).str.zfill(3)
)

df[["RegionName", "StateCodeFIPS", "MunicipalCodeFIPS", "county_fips"]].head(3)

/var/folders/v8/pbmz67g97jd1djd3tpb20jzh0000gn/T/ipykernel_51818/1221318237.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["county_fips"] = (


,RegionName,StateCodeFIPS,MunicipalCodeFIPS,county_fips
0,Los Angeles County,6,37,06037
1,Cook County,17,31,17031
2,Harris County,48,201,48201


In [5]:
# Melt wide (one column per month) -> long (one row per county-month)
date_cols = [c for c in df.columns if c[:4].isdigit() and c.count("-") == 2]
id_cols = [c for c in df.columns if c not in date_cols]

zillow_long = df.melt(
    id_vars=id_cols,
    value_vars=date_cols,
    var_name="date",
    value_name="zhvi",
)
zillow_long["date"] = pd.to_datetime(zillow_long["date"])
zillow_long = zillow_long.dropna(subset=["zhvi"])

print("Long shape:", zillow_long.shape)
print("Unique counties:", zillow_long["county_fips"].nunique())
print("Date range:", zillow_long["date"].min(), "to", zillow_long["date"].max())

Long shape: (708953, 12)
Unique counties: 3071
Date range: 2000-01-31 00:00:00 to 2026-08-31 00:00:00


## 2. Independent Verification of Data Provenance

Reproduces the positive-rate figures from `docs/phase0_framing.md`'s Data Provenance
table independently, using the quarter-end resampling convention (last observed
month's value per county-quarter) and the finalized train/embargo/test split.

Note: since Zillow's file keeps extending forward over time, running this today
(rather than whenever the teammate ran it) will show a later date range and a
larger "Full history" row count than either their original output or the figures
recorded in `docs/phase0_framing.md` — that's expected drift, not a discrepancy to
chase down.

In [6]:
zillow_long["quarter"] = zillow_long["date"].dt.to_period("Q")

zillow_q = (
    zillow_long.sort_values("date")
    .groupby(["county_fips", "quarter"], as_index=False)
    .last()[["county_fips", "quarter", "zhvi"]]
    .sort_values(["county_fips", "quarter"])
    .reset_index(drop=True)
)

zillow_q["pct_change"] = zillow_q.groupby("county_fips")["zhvi"].pct_change()
zillow_q["decline_label"] = (zillow_q["pct_change"] < -0.02).astype(int)
zillow_q_valid = zillow_q.dropna(subset=["pct_change"]).copy()

In [7]:
def window_stats(mask, label):
    sub = zillow_q_valid[mask]
    rate = sub["decline_label"].mean() * 100
    print(f"{label:45s} rows={len(sub):>7,}   positive_rate={rate:.2f}%")

# NOTE: compare Period objects directly, not date strings -- comparing a
# quarter-end Timestamp (23:59:59.999999) against a plain date string silently
# fails, since the string parses to midnight. This bit the teammate once already.
window_stats(pd.Series(True, index=zillow_q_valid.index), "Full history")
window_stats(zillow_q_valid["quarter"] <= pd.Period("2021Q3"), "Train (through 2021-09-30)")
window_stats(zillow_q_valid["quarter"] == pd.Period("2021Q4"), "Embargoed (2021-10-01 to 2021-12-31)")
window_stats(
    (zillow_q_valid["quarter"] >= pd.Period("2022Q1")) & (zillow_q_valid["quarter"] <= pd.Period("2026Q1")),
    "Test (2022-01-01 to 2026-03-31)",
)

Full history                                  rows=235,074   positive_rate=5.59%
Train (through 2021-09-30)                    rows=173,824   positive_rate=4.96%
Embargoed (2021-10-01 to 2021-12-31)          rows=  3,021   positive_rate=0.99%
Test (2022-01-01 to 2026-03-31)               rows= 52,087   positive_rate=8.15%


Expected result (from `docs/phase0_framing.md`, as of when it was last written): Full
history 5.59%; Train 4.96% (173,824 rows); Embargoed 0.99% (3,021 rows); Test 8.15%
(52,087 rows). Train/Embargoed/Test are fixed historical windows, so those three
should match closely even run today. "Full history" will differ since it grows with
each new month Zillow publishes (see note above).

## 3. FRED `MORTGAGE30US` Ingest

Resample from weekly (native FRED resolution) to monthly, using the last observed
weekly rate in each month as that month's value.

Note the resampling convention here differs from `02_fred_ingest_clean.ipynb`, but
only in aggregation, not in month alignment. Both pipelines key to **month-end**
(matching Zillow's grain): this notebook (matching
the teammate's) takes the **last** observed weekly rate in each month, while
`02_fred_ingest_clean.ipynb` is meant to take the **mean** of that month's weekly
rates. `02_fred_ingest_clean.ipynb`'s current code resamples to month-*start*
(`.resample("MS")`) rather than month-end -- that's a bug in that notebook relative
to its own stated intent, not a second legitimate
convention (now fixed there to `"ME"`). Once fixed, the two pipelines differ only
in aggregation (`.last()` vs `.mean()`) -- worth reconciling before either output
feeds Phase 4 feature engineering.


In [8]:
fred = pd.read_csv(FRED_RAW_PATH, parse_dates=["observation_date"])
fred = fred.rename(columns={"observation_date": "date", "MORTGAGE30US": "mortgage30us"})
fred = fred.sort_values("date").reset_index(drop=True)

print("Raw weekly shape:", fred.shape)
print("Missing values:", fred["mortgage30us"].isna().sum())

Raw weekly shape: (2895, 2)
Missing values: 0


In [9]:
fred_monthly = (
    fred.set_index("date")["mortgage30us"]
    .resample("ME")
    .last()
    .reset_index()
)
fred_monthly["month"] = fred_monthly["date"].dt.to_period("M")

print("Monthly shape:", fred_monthly.shape)
print("Missing values after resample:", fred_monthly["mortgage30us"].isna().sum())
fred_monthly.tail(3)

Monthly shape: (666, 3)
Missing values after resample: 0


,date,mortgage30us,month
663,2026-07-31,6.66,2026-07
664,2026-08-31,6.66,2026-08
665,2026-09-30,6.95,2026-09


## 4. Join: Zillow (County, Monthly) + FRED (National, Monthly)

FRED's single national value broadcasts to every county in a given month — a plain
merge on `month` does this automatically, no crosswalk needed.

In [10]:
zillow_long["month"] = zillow_long["date"].dt.to_period("M")

zillow_slim = zillow_long[["county_fips", "RegionName", "month", "zhvi"]].copy()
fred_slim = fred_monthly[["month", "mortgage30us"]].copy()

joined = zillow_slim.merge(fred_slim, on="month", how="left")

print("Joined shape:", joined.shape)
print("Rows with missing mortgage30us:", joined["mortgage30us"].isna().sum())

# Zero-drop check (per docs/phase0_framing.md: no crosswalk needed, so no county should be dropped)
counties_before = set(zillow_slim["county_fips"].unique())
counties_after = set(joined["county_fips"].unique())
dropped = len(counties_before - counties_after)
print("Counties dropped during join:", dropped)
assert dropped == 0, "A county was dropped during the join -- this indicates a bug, not a real geography mismatch."

joined.head(3)

Joined shape: (708953, 5)
Rows with missing mortgage30us: 0
Counties dropped during join: 0


,county_fips,RegionName,month,zhvi,mortgage30us
0,06037,Los Angeles County,2000-01,210217.606330,8.25
1,17031,Cook County,2000-01,152726.258141,8.25
2,48201,Harris County,2000-01,113021.492070,8.25
